<a href="https://colab.research.google.com/github/KerberoSum/Google-LLM-chatroom/blob/main/notebooks/Getting_started_with_google_colab_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import ai
import re
import uuid

# Configuration
selected_model = "google/gemini-2.5-flash-lite"

# --- UI Styling ---
header = widgets.HTML("<h2 style='font-family: Arial;'>💪 Gemini AI Chatroom</h2>")
output_area = widgets.Output(layout=widgets.Layout(height='500px', overflow='scroll', border='1px solid #ddd', padding='10px', margin='10px 0', _dom_classes=[]))

input_box = widgets.Textarea(
    placeholder='Paste your VBA script or message here...',
    layout=widgets.Layout(width='100%', height='150px')
)

send_button = widgets.Button(
    description='Send Message',
    button_style='primary',
    layout=widgets.Layout(width='49%', height='40px'),
    icon='paper-plane'
)

clear_button = widgets.Button(
    description='Clear History',
    button_style='',
    layout=widgets.Layout(width='49%', height='40px'),
    icon='trash'
)

def create_code_block(code_content):
    unique_id = f"code_{uuid.uuid4().hex[:8]}"
    html_str = f"""
    <div style="border: 1px solid #ccc; border-radius: 5px; margin: 10px 0; font-family: Arial;">
        <div style="background: #eee; padding: 5px 10px; display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #ccc;">
            <span style="font-size: 12px; font-weight: bold;">Code Snippet</span>
            <button onclick="navigator.clipboard.writeText(document.getElementById('{unique_id}').innerText); this.innerText='Copied!'; setTimeout(() => {{ this.innerText='Copy'; }}, 2000)"
                    style="cursor: pointer; padding: 2px 8px;">Copy</button>
        </div>
        <pre id="{unique_id}" style="background: #fdfdfd; padding: 10px; margin: 0; white-space: pre-wrap; font-family: 'Courier New', Courier, monospace; overflow-x: auto;">{code_content}</pre>
    </div>
    """
    return widgets.HTML(html_str)

def on_send_clicked(b):
    user_text = input_box.value.strip()
    if not user_text: return
    input_box.value = ''
    with output_area:
        display(widgets.HTML(f"<div style='font-family: Arial;'><b>🙋 You:</b><br/><pre style='white-space: pre-wrap; font-family: Arial;'>{user_text}</pre></div>"))
        display(widgets.HTML("<b style='font-family: Arial;'>✨ AI:</b>"))
        try:
            stream = ai.generate_text(user_text, model_name=selected_model, stream=True)
            full_response = ""
            for chunk in stream: full_response += chunk

            # Split by code blocks
            parts = re.split(r'```(?:[a-zA-Z]*)?\n([\s\S]*?)```', full_response)
            for i, part in enumerate(parts):
                if i % 2 == 1: # This is a code block
                    display(create_code_block(part.strip()))
                else: # Regular text
                    if part.strip():
                        display(widgets.HTML(f"<div style='font-family: Arial; white-space: pre-wrap;'>{part.strip()}</div>"))
            print("-" * 20)
        except Exception as e:
            print(f"\nError: {e}")

def on_clear_clicked(b):
    output_area.clear_output()
    with output_area: print("History cleared.")

send_button.on_click(on_send_clicked)
clear_button.on_click(on_clear_clicked)

control_row = widgets.HBox([send_button, clear_button], layout=widgets.Layout(justify_content='space-between'))
chat_container = widgets.VBox([
    header,
    widgets.HTML(f"<small style='font-family: Arial;'>Model: {selected_model}</small>"),
    output_area, input_box, control_row
], layout=widgets.Layout(width='80%', margin='0 auto'))
display(chat_container)
